In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold

In [ ]:
#import TF dataset
X1=pd.read_csv('Prot-Bert/TF_Training_Embedding_ProtBert.csv', header=None).iloc[:,1:].values
X2=pd.read_csv('Prot-Bert/NTF_Training_Embedding_ProtBert.csv', header=None).iloc[:,1:].values
X_train = np.concatenate((X1,X2),axis=0)

In [ ]:
pos_labels = np.ones(413)
neg_labels = np.zeros(416)
y_train = np.concatenate((pos_labels,neg_labels),axis=0)

In [ ]:
X1=pd.read_csv('Prot-Bert/TF_Ind_Embedding_ProtBert.csv', header=None).iloc[:,1:].values
X2=pd.read_csv('Prot-Bert/NTF_Ind_Embedding_ProtBert.csv', header=None).iloc[:,1:].values
X_test = np.concatenate((X1,X2),axis=0)

In [ ]:
pos_labels = np.ones(106)
neg_labels = np.zeros(106)
y_test = np.concatenate((pos_labels,neg_labels),axis=0)

In [ ]:
print("Shape of the X_train is: ", X_train.shape)
print("Shape of the X_test is: ", X_test.shape)

In [ ]:
def evaluate_model_test(model, X_test, y_test):
    from sklearn import metrics

    # Predict Test Data 
    y_pred = model.predict_proba(X_test)[:,1]
    for i in range(len(y_pred)):
        if y_pred[i]>0.5:
            y_pred[i]=1
        else:
            y_pred[i]=0
    

    # Calculate accuracy, precision, recall, f1-score, and kappa score
    acc = metrics.accuracy_score(y_test, y_pred)
    prec = metrics.precision_score(y_test, y_pred)
    rec = metrics.recall_score(y_test, y_pred)
    f1 = metrics.f1_score(y_test, y_pred)

    # Calculate area under curve (AUC)
    y_pred_proba = model.predict_proba(X_test)[::,1]
    fpr, tpr, _ = metrics.roc_curve(y_test, y_pred_proba)
    auc = metrics.roc_auc_score(y_test, y_pred_proba)
    
    #MCC
    mcc=matthews_corrcoef(y_test, model.predict(X_test))
    
    # Display confussion matrix
    cm = metrics.confusion_matrix(y_test, y_pred)
    total=sum(sum(cm))
    
    #accuracy=(cm[0,0]+cm[1,1])/total
    spec = cm[0,0]/(cm[0,0]+cm[0,1])
    sen= cm[1,1]/(cm[1,0]+cm[1,1])
    
    #print(y_pred_proba)

    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'mcc':mcc,
            'fpr': fpr, 'tpr': tpr, 'auc': auc, 'cm': cm, 'sen': sen, 'spec':spec}

## SVM Classifier and hyperparameter optimization using the Optuna

In [ ]:
import optuna
from optuna.samplers import TPESampler
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
import numpy as np
import gc

seeds = [42, 120, 555]

# Store test results
test_acc_list, test_sn_list, test_sp_list = [], [], []
test_mcc_list, test_auc_list = [], []

for seed in seeds:

    print(f"\n Seed {seed}")

    # CV with seed
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=1, random_state=seed)

    def objective(trial):
        svc_c = trial.suggest_float('C', 1e0, 1e2)
        kernel = trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf'])
        clf = SVC(C=svc_c, kernel=kernel)
        score = cross_val_score(clf, X_train, y_train,cv=cv, scoring="accuracy")
        return score.mean()

    # Seeded Optuna
    sampler = TPESampler(seed=seed)
    svm_study = optuna.create_study(direction='maximize', sampler=sampler)
    svm_study.optimize(objective, n_trials=200)
    print("Best params of the SVM classifier for seed", seed , "is:", svm_study.best_params)

    # Test on the Independent set now
    final_model = SVC(**svm_study.best_params, probability=True)
    final_model.fit(X_train, y_train)
    test_eval = evaluate_model_test(final_model, X_test, y_test)

    print("Test Results on seed ", seed, " are as:")
    print("ACC:", test_eval['acc'],
          "| SN:", test_eval['sen'],
          "| SP:", test_eval['spec'],
          "| MCC:", test_eval['mcc'],
          "| AUC:", test_eval['auc'])

    #Store results
    test_acc_list.append(test_eval['acc'])
    test_sn_list.append(test_eval['sen'])
    test_sp_list.append(test_eval['spec'])
    test_mcc_list.append(test_eval['mcc'])
    test_auc_list.append(test_eval['auc'])

    #Free memory
    del final_model
    del svm_study
    gc.collect()


#Average results 
print("\n The average results on the independent set over three different seeds are as: \n")

print("ACC:", np.mean(test_acc_list))
print("SN :", np.mean(test_sn_list))
print("SP :", np.mean(test_sp_list))
print("MCC:", np.mean(test_mcc_list))
print("AUC:", np.mean(test_auc_list))